In [4]:
import json
import psutil
import platform
import socket
import subprocess

def get_computer_model():
    """Hulpfunctie om het model op te vragen via PowerShell."""
    try:
        cmd = 'powershell "Get-CimInstance -ClassName Win32_ComputerSystem | Select-Object -ExpandProperty Model"'
        output = subprocess.check_output(cmd, shell=True).decode().strip()
        return output if output else "Onbekend"
    except Exception:
        return "Onbekend"

def get_serial_number():
    """Hulpfunctie om het serienummer op te vragen via PowerShell."""
    try:
        cmd = 'powershell "Get-CimInstance -ClassName Win32_BIOS | Select-Object -ExpandProperty SerialNumber"'
        output = subprocess.check_output(cmd, shell=True).decode().strip()
        return output if output else "Onbekend"
    except Exception:
        return "Onbekend"

In [2]:
import json
import psutil
import platform
import socket
import subprocess

def get_computer_model():
    """Hulpfunctie om het model op te vragen via WMI op Windows."""
    try:
        cmd = "wmic csproduct get name"
        output = subprocess.check_output(cmd, shell=True).decode().split('\n')
        return output[1].strip()
    except Exception:
        return "Onbekend"

def collect_system_info():
    """Haalt alle ruwe systeeminformatie op."""
    data = {}

    # CPU info
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    # Memory info
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    # Storage info (alle partities)
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    # Network info
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }

    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }

    return data

def filter_system_info(raw_info):
    """Filtert de ruwe data naar de gevraagde velden."""
    # 1. Bepaal de primaire netwerkinterface
    ignore_list = ["loopback", "vmware", "virtual", "docker", "vethernet", "mcafee"]
    primary_iface = None
    
    for iface_name, iface_data in raw_info["network"]["interfaces"].items():
        if any(ign in iface_name.lower() for ign in ignore_list):
            continue
            
        stats = iface_data.get("stats")
        if stats and stats.get("isup"):
            for addr in iface_data.get("addrs", []):
                if addr.get("family") == 2: # IPv4
                    ip = addr.get("address", "")
                    if not ip.startswith("127.") and not ip.startswith("169.254."):
                        primary_iface = iface_name
                        break
        if primary_iface:
            break

    # Netwerkgegevens uitlezen uit de gekozen interface
    mac_addr = None
    ipv4_addr = None
    ipv6_addr = None
    ipv6_link_local = None

    if primary_iface and primary_iface in raw_info["network"]["interfaces"]:
        for addr in raw_info["network"]["interfaces"][primary_iface]["addrs"]:
            fam = addr.get("family")
            ip = addr.get("address", "")
            
            if fam == -1:  # MAC
                mac_addr = ip
            elif fam == 2:  # IPv4
                ipv4_addr = ip
            elif fam == 23 or ":" in ip:  # IPv6
                if ip.lower().startswith("fe80"):
                    ipv6_link_local = ip
                else:
                    ipv6_addr = ip

    # 2. Bepaal opslag (Primaire OS-schijf, bijv. C:\ of /)
    storage_dict = raw_info.get("storage", {})
    os_drive_key = "C:\\" if "C:\\" in storage_dict else (list(storage_dict.keys())[0] if storage_dict else None)
    
    if os_drive_key and os_drive_key in storage_dict:
        drive = storage_dict[os_drive_key]
        fstype = drive.get("fstype", "N/A")
        capacity_gb = round(drive.get("total", 0) / (1024**3), 2)
    else:
        fstype = "N/A"
        capacity_gb = 0.0

    # 3. CPU Freq & RAM omrekenen
    cpu_freq = raw_info["cpu"].get("cpu_freq")
    max_freq = cpu_freq.get("max") if cpu_freq else "N/A"
    ram_gb = round(raw_info["memory"]["total"] / (1024**3), 2)

    # 4. Nieuwe gefilterde dictionary samenstellen
    filtered_dict = {
        "computernaam": raw_info["network"]["hostname"],
        "computermodel": get_computer_model(),
        "processor": raw_info["cpu"]["processor"],
        "aantal_fysieke_cores": raw_info["cpu"]["physical_cores"],
        "max_cpu_frequentie_mhz": max_freq,
        "totale_ram_gb": ram_gb,
        "bestandssysteem_opslag": fstype,
        "opslagcapaciteit_gb": capacity_gb,
        "mac_adres": mac_addr,
        "ipv4_adres": ipv4_addr,
        "ipv6_adres": ipv6_addr,
        "ipv6_link_local": ipv6_link_local
    }

    return filtered_dict


# --- DIT BLOK STAAT NUL SPATIES INGESPRONGEN (TEGEN DE KANT): ---
if __name__ == "__main__":
    # De regels hieronder zijn EXACT 4 SPATIES ingesprongen:
    raw_data = collect_system_info()
    gefilterde_data = filter_system_info(raw_data)
    print(json.dumps(gefilterde_data, indent=4))

{
    "computernaam": "LaptopPallieter",
    "computermodel": "Onbekend",
    "processor": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
    "aantal_fysieke_cores": 10,
    "max_cpu_frequentie_mhz": 2400.0,
    "totale_ram_gb": 31.71,
    "bestandssysteem_opslag": "NTFS",
    "opslagcapaciteit_gb": 454.7,
    "mac_adres": "0A-00-27-00-00-16",
    "ipv4_adres": "192.168.56.1",
    "ipv6_adres": null,
    "ipv6_link_local": "fe80::e07f:53e1:4496:2cfe"
}


In [3]:
def filter_system_info(raw_info):
    """Filtert de ruwe data naar de gevraagde velden."""
    # 1. Bepaal de primaire netwerkinterface
    ignore_list = ["loopback", "vmware", "virtual", "docker", "vethernet", "mcafee"]
    primary_iface = None
    
    for iface_name, iface_data in raw_info["network"]["interfaces"].items():
        if any(ign in iface_name.lower() for ign in ignore_list):
            continue
            
        stats = iface_data.get("stats")
        if stats and stats.get("isup"):
            for addr in iface_data.get("addrs", []):
                if addr.get("family") == 2: # IPv4
                    ip = addr.get("address", "")
                    if not ip.startswith("127.") and not ip.startswith("169.254."):
                        primary_iface = iface_name
                        break
        if primary_iface:
            break

    # Netwerkgegevens uitlezen uit de gekozen interface
    mac_addr = None
    ipv4_addr = None
    ipv6_addr = None
    ipv6_link_local = None

    if primary_iface and primary_iface in raw_info["network"]["interfaces"]:
        for addr in raw_info["network"]["interfaces"][primary_iface]["addrs"]:
            fam = addr.get("family")
            ip = addr.get("address", "")
            
            if fam == -1:  # MAC
                mac_addr = ip
            elif fam == 2:  # IPv4
                ipv4_addr = ip
            elif fam == 23 or ":" in ip:  # IPv6
                if ip.lower().startswith("fe80"):
                    ipv6_link_local = ip
                else:
                    ipv6_addr = ip

    # 2. Bepaal opslag (Primaire OS-schijf)
    storage_dict = raw_info.get("storage", {})
    os_drive_key = "C:\\" if "C:\\" in storage_dict else (list(storage_dict.keys())[0] if storage_dict else None)
    
    if os_drive_key and os_drive_key in storage_dict:
        drive = storage_dict[os_drive_key]
        fstype = drive.get("fstype", "N/A")
        capacity_gb = round(drive.get("total", 0) / (1024**3), 2)
    else:
        fstype = "N/A"
        capacity_gb = 0.0

    # 3. CPU Freq & RAM omrekenen
    cpu_freq = raw_info["cpu"].get("cpu_freq")
    max_freq = cpu_freq.get("max") if cpu_freq else "N/A"
    ram_gb = round(raw_info["memory"]["total"] / (1024**3), 2)

    # 4. Nieuwe gefilterde dictionary samenstellen
    filtered_dict = {
        "computernaam": raw_info["network"]["hostname"],
        "computermodel": get_computer_model(),
        "serienummer": get_serial_number(),
        "processor": raw_info["cpu"]["processor"],
        "aantal_fysieke_cores": raw_info["cpu"]["physical_cores"],
        "max_cpu_frequentie_mhz": max_freq,
        "totale_ram_gb": ram_gb,
        "bestandssysteem_opslag": fstype,
        "opslagcapaciteit_gb": capacity_gb,
        "mac_adres": mac_addr,
        "ipv4_adres": ipv4_addr,
        "ipv6_adres": ipv6_addr,
        "ipv6_link_local": ipv6_link_local
    }

    return filtered_dict

# Data verzamelen en filteren
raw_data = collect_system_info()
gefilterde_data = filter_system_info(raw_data)

# Print het resultaat netjes op het scherm
print(json.dumps(gefilterde_data, indent=4))

{
    "computernaam": "LaptopPallieter",
    "computermodel": "Cyborg 15 A13VFK",
    "serienummer": "Onbekend",
    "processor": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
    "aantal_fysieke_cores": 10,
    "max_cpu_frequentie_mhz": 2400.0,
    "totale_ram_gb": 31.71,
    "bestandssysteem_opslag": "NTFS",
    "opslagcapaciteit_gb": 454.7,
    "mac_adres": "0A-00-27-00-00-16",
    "ipv4_adres": "192.168.56.1",
    "ipv6_adres": null,
    "ipv6_link_local": "fe80::e07f:53e1:4496:2cfe"
}


In [4]:
import csv

csv_filename = "systeeminformatie.csv"
fieldnames = list(gefilterde_data.keys())

# Let op de komma na newline=""
with open(csv_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=";")
    writer.writeheader()
    writer.writerow(gefilterde_data)

print(f"CSV-bestand '{csv_filename}' succesvol aangemaakt.")

CSV-bestand 'systeeminformatie.csv' succesvol aangemaakt.


In [5]:
!pip install openpyxl


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import openpyxl

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Systeeminformatie"

headers = list(gefilterde_data.keys())
ws.append(headers)

values = list(gefilterde_data.values())
ws.append(values)

wb.save("systeeminformatie.xlsx")
print("Spreadsheet 'systeeminformatie.xlsx' succesvolaangemaakt.")

Spreadsheet 'systeeminformatie.xlsx' succesvolaangemaakt.


In [7]:
import os
import csv

input_folder = "."  # "." betekent de huidige map waar je notebook staat
output_csv = "verzamelde_systeeminformatie.csv"

alle_rijen = []
fieldnames = None

for file in os.listdir(input_folder):
    # Verwerk alleen CSV-bestanden, maar sla de samengevoegde CSV op te voorkomen dat je in een lus blijft hangen
    if file.endswith(".csv") and file != output_csv:
        file_path = os.path.join(input_folder, file)
        with open(file_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=";")
            fieldnames = reader.fieldnames
            for row in reader:
                alle_rijen.append(row)

if fieldnames and alle_rijen:
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=";")
        writer.writeheader()
        writer.writerows(alle_rijen)
    print(f"Dataset succesvol samengevoegd in '{output_csv}'. Totaal aantal computers: {len(alle_rijen)}")

Dataset succesvol samengevoegd in 'verzamelde_systeeminformatie.csv'. Totaal aantal computers: 1


In [8]:
import csv
import re

input_csv = "verzamelde_systeeminformatie.csv"

# Reguliere expressies voor patronen
ipv4_pattern = r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$"
mac_pattern = r"^([0-9A-Fa-f]{2}[:-]){5}([0-9A-Fa-f]{2})$"

print("--- START AUTOMATISCHE VALIDATIE ---\n")

with open(input_csv, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    
    for i, rij in enumerate(reader, start=1):
        computer = rij.get("computernaam", f"Computer {i}")
        print(f"Controleren van: {computer}")
        
        # 1. Check op lege/ontbrekende velden
        for key, value in rij.items():
            if not value or value == "N/A" or value == "None":
                print(f"  ⚠️ Waarschuwing: Veld '{key}' is leeg of onbekend.")
        
        # 2. RAM & Cores controle (moeten positieve getallen zijn)
        try:
            ram = float(rij.get("totale_ram_gb", 0))
            if ram <= 0:
                print("  ❌ Fout: RAM-geheugen moet groter zijn dan 0 GB.")
        except ValueError:
            print("  ❌ Fout: RAM-geheugen is geen geldig getal.")
            
        try:
            cores = int(rij.get("aantal_fysieke_cores", 0))
            if cores < 1:
                print("  ❌ Fout: Aantal fysieke cores moet minstens 1 zijn.")
        except ValueError:
            print("  ❌ Fout: Aantal cores is geen geldig geheel getal.")

        # 3. IPv4-adres validatie
        ipv4 = rij.get("ipv4_adres", "")
        if not re.match(ipv4_pattern, ipv4):
            print(f"  ❌ Fout: Ongeldig IPv4-adres formaat: '{ipv4}'")

        # 4. MAC-adres validatie
        mac = rij.get("mac_adres", "")
        if not re.match(mac_pattern, mac):
            print(f"  ❌ Fout: Ongeldig MAC-adres formaat: '{mac}'")
            
        print("  ----")

print("\n--- VALIDATIE VOLTOOID ---")

--- START AUTOMATISCHE VALIDATIE ---

Controleren van: LaptopPallieter
  ⚠️ Waarschuwing: Veld 'ipv6_adres' is leeg of onbekend.
  ----

--- VALIDATIE VOLTOOID ---
